In [1]:
import pandas as pd
import numpy as np

honey = pd.read_csv("honey.csv", sep=";")
beehives = pd.read_csv("ule.csv", sep=";")
hourly = pd.read_csv("df_hourly_2019_2025.csv")

In [3]:
hourly.head()


,valid_time,temp,wspd,prcp,pres,week
0,2019-05-01 00:00:00,8.404327,3.310282,0.178814,941.63280,18
1,2019-05-01 01:00:00,8.166046,3.494877,0.156403,941.33044,18
2,2019-05-01 02:00:00,7.969513,3.636947,0.191212,941.26220,18
3,2019-05-01 03:00:00,7.678314,3.815005,0.128269,940.87940,18
4,2019-05-01 04:00:00,7.543060,3.679924,0.025272,940.96716,18


In [4]:

hourly.info()

<class 'pandas.DataFrame'>
RangeIndex: 25704 entries, 0 to 25703
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   valid_time  25704 non-null  str    
 1   temp        25704 non-null  float64
 2   wspd        25704 non-null  float64
 3   prcp        25704 non-null  float64
 4   pres        25704 non-null  float64
 5   week        25704 non-null  int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 1.2 MB


In [5]:
hourly["valid_time"] = pd.to_datetime(hourly["valid_time"])

hourly["year"] = hourly["valid_time"].dt.year
hourly["month"] = hourly["valid_time"].dt.month
hourly["day"] = hourly["valid_time"].dt.day

hourly.head()

,valid_time,temp,wspd,prcp,pres,week,year,month,day
0,2019-05-01 00:00:00,8.404327,3.310282,0.178814,941.63280,18,2019,5,1
1,2019-05-01 01:00:00,8.166046,3.494877,0.156403,941.33044,18,2019,5,1
2,2019-05-01 02:00:00,7.969513,3.636947,0.191212,941.26220,18,2019,5,1
3,2019-05-01 03:00:00,7.678314,3.815005,0.128269,940.87940,18,2019,5,1
4,2019-05-01 04:00:00,7.543060,3.679924,0.025272,940.96716,18,2019,5,1


In [6]:
print(hourly["valid_time"].min())
print(hourly["valid_time"].max())

hourly.groupby("year").size()

2019-05-01 00:00:00
2025-09-30 23:00:00


year
2019    3672
2020    3672
2021    3672
2022    3672
2023    3672
2024    3672
2025    3672
dtype: int64

In [7]:
def make_same_week_hourly_features(row, hourly):
    year = row["year"]
    week = row["week"]

    w = hourly[
        (hourly["year"] == year) &
        (hourly["week"] == week)
    ]

    return pd.Series({
        "temp_mean": w["temp"].mean(),
        "temp_min": w["temp"].min(),
        "temp_max": w["temp"].max(),
        "prcp_sum": w["prcp"].sum(),
        "rain_hours": (w["prcp"] > 0).sum(),
        "wspd_mean": w["wspd"].mean(),
        "wspd_max": w["wspd"].max(),
        "pres_mean": w["pres"].mean()
    })

In [8]:
hourly_features = honey.apply(
    lambda row: make_same_week_hourly_features(row, hourly),
    axis=1
)

df_hourly_model = pd.concat([honey, hourly_features], axis=1)
df_hourly_model = df_hourly_model.merge(beehives, on="year", how="left")

df_hourly_model.head()

,year,week,honey,harvest,temp_mean,temp_min,temp_max,prcp_sum,rain_hours,wspd_mean,wspd_max,pres_mean,beehive
0,2019,30,66.0,1,18.468962,12.626312,28.296112,21.147251,100.0,2.068542,4.798727,949.431772,92
1,2019,32,55.0,1,19.387705,10.320038,28.937897,28.769016,83.0,1.972191,4.342981,948.638158,92
2,2019,33,134.0,2,16.821316,7.273743,29.304504,18.177986,87.0,1.924085,4.590142,949.983344,92
3,2019,34,72.0,1,18.534908,10.699921,29.448029,5.413055,59.0,1.906017,4.757523,957.323854,92
4,2019,35,59.0,1,20.576924,14.233734,29.156158,11.940956,68.0,1.698392,3.565312,954.631368,92


In [9]:
df_hourly_model.shape

(47, 13)

In [10]:
df_hourly_direct = honey.merge(
    hourly,
    on=["year", "week"],
    how="inner"
)

df_hourly_direct = df_hourly_direct.merge(
    beehives,
    on="year",
    how="left"
)

df_hourly_direct.head()

,year,week,honey,harvest,valid_time,temp,wspd,prcp,pres,month,day,beehive
0,2019,30,66.0,1,2019-07-22 00:00:00,13.695709,2.348979,0.0,955.99580,7,22,92
1,2019,30,66.0,1,2019-07-22 01:00:00,13.675629,2.642256,0.0,955.93560,7,22,92
2,2019,30,66.0,1,2019-07-22 02:00:00,13.677185,2.920277,0.0,955.90656,7,22,92
3,2019,30,66.0,1,2019-07-22 03:00:00,13.855621,2.831057,0.0,955.89233,7,22,92
4,2019,30,66.0,1,2019-07-22 04:00:00,14.106293,2.649939,0.0,956.11520,7,22,92


In [11]:
df_hourly_direct.shape

(7896, 12)

In [12]:
df_hourly_direct["valid_time"] = pd.to_datetime(df_hourly_direct["valid_time"])

df_hourly_direct["hour"] = df_hourly_direct["valid_time"].dt.hour
df_hourly_direct["dayofweek"] = df_hourly_direct["valid_time"].dt.dayofweek

In [13]:
features = [
    "year",
    "week",
    "harvest",
    "beehive",
    "temp",
    "wspd",
    "prcp",
    "pres",
    "hour",
    "dayofweek"
]

X = df_hourly_direct[features]
y = df_hourly_direct["honey"]
groups = df_hourly_direct["year"]

In [14]:
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
import pandas as pd

cv = LeaveOneGroupOut()

models = {
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=15.615096158372753))
    ]),

    "ElasticNet": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=4.3981810937396855,
            l1_ratio=0.997928523653379,
            max_iter=10000,
            random_state=42
        ))
    ])
}

results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv.split(X, y, groups),
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error"
        }
    )

    results.append({
        "model": name,
        "MAE": -scores["test_mae"].mean(),
        "RMSE": -scores["test_rmse"].mean()
    })

results_hourly_direct = pd.DataFrame(results).sort_values("MAE")
results_hourly_direct

,model,MAE,RMSE
1,ElasticNet,58.761642,71.932958
0,Ridge,62.476229,76.990354
